In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
from util import split_wav_by_timestamps, aggregate_timestamps

token = os.environ.get('HUGGINGFACE_ACCESS_TOKEN')
wav_file = "sample.wav"

In [6]:
from pyannote.audio import Pipeline
import torch
import torchaudio

pipeline = Pipeline.from_pretrained(
  "pyannote/speaker-diarization-3.1",
  use_auth_token=token)

pipeline.to(torch.device("cuda"))

waveform, sample_rate = torchaudio.load(wav_file)

diarization = pipeline({"waveform": waveform, "sample_rate": sample_rate}, num_speakers=2)

In [7]:
timestamps = []

for turn, _, speaker in diarization.itertracks(yield_label=True):
    timestamps.append((turn.start, turn.end, speaker))

In [10]:
output_dir = 'out'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

split_wav_by_timestamps(wav_file, output_dir, timestamps, 'timestamp')